# 03 — Transform (Incremental Silver → Gold)
ADF passes the same `batch_id`; only that Silver batch is transformed and Gold is upserted by URL.


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

dbutils.widgets.text("batch_id", "")
batch_id = dbutils.widgets.get("batch_id").strip()
if not batch_id:
    raise ValueError("batch_id is required")

BASE_PATH = "abfss://edtech@edtechpipline26.dfs.core.windows.net"
VALIDATED_PATH = f"{BASE_PATH}/interim/validated"
FINAL_PATH = f"{BASE_PATH}/processed/final"

validated_batch = (spark.read.format("delta").load(VALIDATED_PATH)
                   .filter(F.col("batch_id") == batch_id))


In [0]:
final_batch = (validated_batch
    .withColumn("author",F.when(F.col("author").isNull() | (F.trim(F.col("author"))==""),
                                F.lit("Unknown")).otherwise(F.trim(F.col("author"))))
    .withColumn("source",F.trim(F.col("source")))
    .withColumn("category",F.trim(F.col("category")))
    .withColumn("title",F.trim(F.col("title")))
    .withColumn("url",F.trim(F.col("url")))
    .withColumn("word_count",
        F.when(F.col("content").isNull() | (F.trim(F.col("content"))==""),F.lit(0))
         .otherwise(F.size(F.split(F.trim(F.col("content")),r"\s+"))))
    .withColumn("publish_year",F.year(F.col("publication_date")))
    .withColumn("is_long_form",F.col("word_count") > 500))


In [0]:
if DeltaTable.isDeltaTable(spark, FINAL_PATH):
    (DeltaTable.forPath(spark, FINAL_PATH).alias("t")
     .merge(final_batch.alias("s"), "t.url = s.url")
     .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
else:
    final_batch.write.format("delta").mode("overwrite").save(FINAL_PATH)
